# 08 Preprocessing validation

Validate the cleaned release, audited features, frozen split and fitted preprocessor. Reproduce a second fit on training features alone. Save the cleaned and engineered rows, transformed inputs and separate keyed labels for each split. Run the automated tests on the resulting files. No model is trained.

In [ ]:
from hashlib import file_digest
from pathlib import Path
from tempfile import TemporaryDirectory
import json
import platform
import subprocess
import sys

import joblib
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from scipy import sparse
import sklearn
import yaml
from IPython.display import display

current = Path.cwd()
root = next(path for path in (current, *current.parents) if (path / "AGENTS.md").is_file())
sys.path.insert(0, str(root))
sys.dont_write_bytecode = True

from src.features import FEATURE_COLUMNS, HISTORY_FEATURES, OUTPUT_COLUMNS, build_features
from src.integration import (
    BATCH_READ_COLUMNS, ENTITY_READ_COLUMNS, MEDICINE_READ_COLUMNS, PREDICTION_TIMESTAMP,
    TRANSACTION_KEY, TRANSACTION_READ_COLUMNS, integrate_transactions, require_unique_key,
)
from src.preprocessing import SPLIT_NAMES, TARGET_COLUMN, align_target, index_targets, make_preprocessor, training_class_weights

processed = root / "processed"
cleaned = processed / "cleaned"
final = processed / "final"
artifacts = root / "artifacts"
reports = root / "reports"
boundary = json.loads((processed / "split_boundaries.json").read_text(encoding="utf-8"))
manifest_path = processed / boundary["assignment_file"]
preprocessor_path = artifacts / "preprocessor.joblib"
names_path = artifacts / "final_feature_names.json"
weights_path = artifacts / "training_class_weights.json"

def sha256(path):
    with path.open("rb") as source:
        return file_digest(source, "sha256").hexdigest()

assert sha256(manifest_path) == boundary["assignment_sha256"]
for relative, fingerprint in boundary["input_sha256"].items():
    assert sha256(root / relative) == fingerprint

key = list(TRANSACTION_KEY)
clock = PREDICTION_TIMESTAMP
identity = [*key, clock]
order = [clock, *key]
assert key == boundary["transaction_key"] and clock == boundary["prediction_timestamp"]
assert TARGET_COLUMN == boundary["target_column"]
audit = (reports / "leakage_audit.md").read_text(encoding="utf-8")
assert all(f"| `{name}` |" in audit for name in FEATURE_COLUMNS)
assert set(FEATURE_COLUMNS).isdisjoint({"batch_id", clock, "split", TARGET_COLUMN})

cleaning_counts = {}
for name in ("entities", "medicines", "batches", "fraud_events", "transactions"):
    filename = f"{name}.parquet"
    raw = pq.read_metadata(root / "Phase1_Dataset" / "dataset" / filename)
    clean = pq.read_metadata(cleaned / filename)
    assert raw.num_rows == clean.num_rows and raw.num_columns == clean.num_columns
    cleaning_counts[name] = clean.num_rows
display(pd.Series(cleaning_counts, name="unchanged cleaned row count").to_frame())

## Rebuild the audited rows

Check unique keys and row counts at each join. Match the frozen assignment on the complete transaction key and timestamp. Verify batch and timestamp separation across the retained splits and keep the existing boundary exclusions. The checks work without reading any labels.

In [ ]:
def read_cleaned(name, columns):
    return pd.read_parquet(cleaned / f"{name}.parquet", columns=list(columns))


config = yaml.safe_load((root / "Phase1_Dataset/code/config.yaml").read_text(encoding="utf-8"))
integrated, join_audit, timing = integrate_transactions(
    read_cleaned("transactions", TRANSACTION_READ_COLUMNS),
    read_cleaned("batches", BATCH_READ_COLUMNS),
    read_cleaned("medicines", MEDICINE_READ_COLUMNS),
    read_cleaned("entities", ENTITY_READ_COLUMNS),
    config["storage_ranges"],
)
assert all(row["rows_before"] == row["rows_after"] and row["unmatched_foreign_keys"] == 0 for row in join_audit)
assert timing["storage_rate_matches"] == len(integrated) and timing["preproduction_transactions"] == 0
manifest = pd.read_parquet(manifest_path, columns=[*identity, "split"])
require_unique_key(manifest, key, "frozen allocation")
assigned = integrated.merge(manifest, on=identity, how="inner", validate="one_to_one")
assert len(assigned) == len(manifest) == len(integrated) == boundary["source_rows"]
assert assigned.groupby("batch_id")["split"].nunique().eq(1).all()
assert set(assigned["split"]) == {*SPLIT_NAMES, boundary["excluded_assignment"]}
del manifest, integrated

retained = assigned.loc[assigned["split"].isin(SPLIT_NAMES)]
assert len(retained) == boundary["retained_rows"]
excluded = len(assigned) - len(retained)
assert excluded == boundary["split_summary"][boundary["excluded_assignment"]]["rows"]
del assigned

ordered = retained[[*identity, "split"]].sort_values(order, ignore_index=True)
engineered = build_features(retained)
pd.testing.assert_frame_equal(engineered[identity], ordered[identity])
assert list(engineered.columns) == list(OUTPUT_COLUMNS)
require_unique_key(engineered, key, "retained engineered events")
assert engineered[clock].is_monotonic_increasing
del retained

split_index = {}
previous_end = None
for name in SPLIT_NAMES:
    rows = np.flatnonzero(ordered["split"].eq(name))
    assert len(rows) == boundary["split_summary"][name]["rows"]
    assert rows[-1] - rows[0] + 1 == len(rows), "A split is interleaved with another split"
    split_index[name] = slice(rows[0], rows[-1] + 1)

    dates = engineered.iloc[split_index[name]][clock]
    expected = boundary["split_summary"][name]
    assert dates.iloc[0] == pd.Timestamp(expected["first_timestamp"])
    assert dates.iloc[-1] == pd.Timestamp(expected["last_timestamp"])
    if previous_end is not None:
        assert previous_end < dates.iloc[0]
    previous_end = dates.iloc[-1]

assert engineered.iloc[split_index["train"]][clock].max() < pd.Timestamp(boundary["validation_start_inclusive"])
assert engineered.iloc[split_index["validation"]][clock].max() < pd.Timestamp(boundary["test_start_inclusive"])
del ordered
print(f"Verified {len(engineered):,} retained keys and {excluded:,} excluded rows against the frozen split.")

## Verify mathematical and point-in-time behavior

Independently recompute every prior event count and use direct filtering to verify elapsed times, means and rates on a fixed set of rows from every split. Check that raw history is missing only before a batch has an earlier event. Repeat the fit on training features, then compare its complete fitted state to the existing pipeline. Loaded validation and test features are never passed to a fit call.

In [ ]:
prior_counts = engineered.groupby("batch_id", sort=False).cumcount()
prior_counts = prior_counts - engineered.groupby(["batch_id", clock], sort=False).cumcount()
assert engineered["prior_event_count"].eq(prior_counts).all()
no_history = prior_counts.eq(0)
history_values = list(HISTORY_FEATURES[1:])
assert engineered[history_values].isna().eq(no_history, axis=0).all().all()
assert engineered.drop(columns=history_values).notna().all().all()
assert not np.isinf(engineered.select_dtypes(include="number").to_numpy()).any()

groups = engineered.groupby("batch_id", sort=False).indices
compared_history_rows = 0
for name in SPLIT_NAMES:
    bounds = split_index[name]
    positions = np.linspace(bounds.start, bounds.stop - 1, num=65, dtype=int)
    for position in positions:
        row = engineered.iloc[position]
        same_batch = engineered.iloc[groups[row["batch_id"]]]
        before = same_batch.loc[same_batch[clock].lt(row[clock])]
        assert len(before) == row["prior_event_count"]
        if not before.empty:
            expected = [
                (row[clock] - before[clock].max()).total_seconds(),
                (row[clock] - before[clock].min()).total_seconds(),
                before["humidity"].mean(), before["weight"].mean(),
                before["rfid_scanned"].mean(), before["door_open"].mean(),
            ]
            np.testing.assert_allclose(row[history_values].to_numpy(dtype=float), expected, rtol=1e-12, atol=1e-12)
        compared_history_rows += 1
del groups

features = list(FEATURE_COLUMNS)
fitted = joblib.load(preprocessor_path)
output_names = json.loads(names_path.read_text(encoding="utf-8"))
assert fitted.feature_names_in_.tolist() == features
assert fitted.get_feature_names_out().tolist() == output_names
allowed = {"numeric", "categorical", "remainder"}
assert {name for name, _, _ in fitted.transformers_} <= allowed
assert fitted.remainder == "drop" and len(output_names) == len(set(output_names))
forbidden = {"batch_id", clock, "split", TARGET_COLUMN, "fraud_type", "is_fraud_detected",
             "is_masked", "is_suspicious_genuine", "in_ring", "temperature", "seal_intact", "qr_verified",
             "gps_lat", "gps_lng", "rolling_excursion_rate", "duplicate_serial", "cumulative_excursion_time"}
assert set(features).isdisjoint(forbidden)
assert not any(name.split("__", 1)[-1] in forbidden for name in output_names)

train_raw = engineered.iloc[split_index["train"]][features]
repeat = make_preprocessor().fit(train_raw)
fitted_state = joblib.hash(fitted)
assert joblib.hash(repeat) == fitted_state
assert np.all(fitted.named_transformers_["numeric"]["scale"].n_samples_seen_ == len(train_raw))
del repeat

def to_dense(matrix):
    return matrix.toarray() if sparse.issparse(matrix) else matrix


probe = train_raw.iloc[:1]
event_columns = [i for i, name in enumerate(output_names) if name.startswith("categorical__event_type_")]
assert event_columns
unseen = to_dense(fitted.transform(probe.assign(event_type="__unseen_for_pipeline_check__")))
assert np.isfinite(unseen).all() and np.count_nonzero(unseen[:, event_columns]) == 0
assert joblib.hash(fitted) == fitted_state
del train_raw
print(f"Reproduced the training-only fit and independently verified history on {compared_history_rows} rows.")

## Save aligned data and verify the written files

The engineered and target files both carry transaction keys. The model-ready input file contains only the 26 transformed columns. All three files for a split have the same chronological row order. The manifest records file checksums, the feature names and the fitted pipeline checksum.

Expected first-event history gaps stay visible in the engineered files for auditing. The fitted training transformer fills those gaps for the model-ready files. All written feature values and labels are checked against the in-memory results.

In [ ]:
target_table = pd.read_parquet(processed / "transaction_target.parquet", columns=[*key, TARGET_COLUMN])
assert len(target_table) == boundary["source_rows"]
indexed = index_targets(target_table)
del target_table
initial_weights = json.loads(weights_path.read_text(encoding="utf-8"))
assert initial_weights["split_assignment_sha256"] == boundary["assignment_sha256"]

# Re-save the verified pipeline and check that loading the file preserves its state.
with TemporaryDirectory(dir=artifacts) as folder:
    staged_pipeline = Path(folder) / preprocessor_path.name
    joblib.dump(fitted, staged_pipeline)
    assert joblib.hash(joblib.load(staged_pipeline)) == fitted_state
    staged_pipeline.replace(preprocessor_path)

final.mkdir(exist_ok=True)
block_size = 50_000
saved_splits = {}
with TemporaryDirectory(dir=final) as folder:
    staging = Path(folder)
    filenames = []
    for name in SPLIT_NAMES:
        rows = engineered.iloc[split_index[name]].reset_index(drop=True)
        labels = align_target(rows[key], indexed)
        transformed = fitted.transform(rows[features])
        assert joblib.hash(fitted) == fitted_state
        assert transformed.shape == (len(rows), len(output_names)) and labels.shape == (len(rows),)
        assert transformed.dtype == np.float32 and labels.dtype == np.uint8
        genuine, fraud = map(int, np.bincount(labels, minlength=2))
        expected = boundary["split_summary"][name]
        assert (genuine, fraud) == (expected["genuine_rows"], expected["fraud_rows"])
        if name == "train":
            assert training_class_weights(labels) == {int(k): w for k, w in initial_weights["class_weights"].items()}

        engineered_file = staging / f"{name}_engineered.parquet"
        matrix_file = staging / f"{name}_X.parquet"
        target_file = staging / f"{name}_target.parquet"
        rows.to_parquet(engineered_file, index=False, compression="zstd", row_group_size=block_size)
        keyed_labels = rows[key].assign(**{TARGET_COLUMN: labels})
        keyed_labels.to_parquet(target_file, index=False, compression="zstd", row_group_size=block_size)
        pd.testing.assert_frame_equal(pd.read_parquet(target_file), keyed_labels, check_exact=True)

        schema = pa.schema([(column, pa.float32()) for column in output_names])
        with pq.ParquetWriter(matrix_file, schema, compression="zstd") as writer:
            for start in range(0, len(rows), block_size):
                block = to_dense(transformed[start:start + block_size])
                assert np.isfinite(block).all()
                writer.write_table(pa.table(dict(zip(output_names, block.T, strict=True)), schema=schema))
        with pq.ParquetFile(matrix_file) as matrix, pq.ParquetFile(engineered_file) as exported:
            assert matrix.schema_arrow == schema and matrix.metadata.num_rows == exported.metadata.num_rows == len(rows)
            offset = 0
            for x_batch, raw_batch in zip(
                matrix.iter_batches(batch_size=block_size), exported.iter_batches(batch_size=block_size), strict=True,
            ):
                end = offset + len(x_batch)
                np.testing.assert_array_equal(x_batch.to_pandas().to_numpy(), to_dense(transformed[offset:end]))
                pd.testing.assert_frame_equal(raw_batch.to_pandas(), rows.iloc[offset:end].reset_index(drop=True),
                                              check_dtype=False, check_exact=True)
                offset = end
            assert offset == len(rows)

        files = {"engineered": engineered_file, "matrix": matrix_file, "target": target_file}
        filenames.extend(files.values())
        saved_splits[name] = {
            "rows": len(rows), "first_timestamp": rows[clock].iloc[0].isoformat(),
            "last_timestamp": rows[clock].iloc[-1].isoformat(),
            "genuine": genuine, "fraud": fraud, "rows_without_prior_history": int(rows["prior_event_count"].eq(0).sum()),
            "files": {kind: {"name": path.name, "sha256": sha256(path)} for kind, path in files.items()},
        }
        del rows, labels, transformed, keyed_labels

    delivered = {
        "split_assignment_sha256": boundary["assignment_sha256"],
        "preprocessor_sha256": sha256(preprocessor_path),
        "transaction_key": key,
        "row_order": order,
        "input_features": features,
        "output_features": output_names,
        "excluded_rows": excluded,
        "software_versions": {"python": platform.python_version(), "scikit-learn": sklearn.__version__,
                              "pandas": pd.__version__, "pyarrow": pa.__version__, "numpy": np.__version__,
                              "joblib": joblib.__version__},
        "splits": saved_splits,
    }
    staged_manifest = staging / "manifest.json"
    staged_manifest.write_text(json.dumps(delivered, indent=2, allow_nan=False) + "\n", encoding="utf-8")
    for path in [*filenames, staged_manifest]:
        path.replace(final / path.name)

assert joblib.hash(joblib.load(preprocessor_path)) == fitted_state
print("Saved and round-trip verified nine Parquet files and the fitted preprocessor.")

## Execute automated tests and record results

Run the future-invariance tests, training-only transformation tests, and final dataset tests. The report records the actual test result and split distributions. Held-out label counts are reported for validation only and did not influence the fitted state.

In [ ]:
test_run = subprocess.run(
    [sys.executable, "-B", "-m", "pytest", "-q", "-p", "no:cacheprovider", "tests"],
    cwd=root, capture_output=True, text=True,
)
print(test_run.stdout, end="")
if test_run.stderr:
    print(test_run.stderr, end="")
assert test_run.returncode == 0, "See the failure output above."

result_rows = pd.DataFrame(saved_splits).T[["rows", "genuine", "fraud", "rows_without_prior_history"]]
result_rows["transformed columns"] = len(output_names)
display(result_rows)
print("All final validation assertions passed.")

In [ ]:
report = [
    "# Preprocessing validation report",
    f"The [validation notebook](../notebooks/08_Preprocessing_Validation.ipynb) verified and saved "
    f"{sum(item['rows'] for item in saved_splits.values()):,} retained transactions across three splits. "
    "No model was trained. [Output manifest](../processed/final/manifest.json)",
    "## Decisions",
    "- The prediction unit is a transaction with key `(batch_id, leg, event_type)`. The recorded `timestamp` "
    "is the offline scoring clock. The documented batch-level fraud decision is copied to transactions and "
    "saved separately as a binary target. [Prediction design](prediction_design.md)\n"
    "- The frozen split keeps batches and matching timestamps together across the retained windows. "
    f"It excludes {excluded:,} rows from 1,536 boundary-spanning batches. All later checks use the same assignments.\n"
    "- Cleaning uses fixed schemas and categories, preserves expected source nulls and valid anomalies, and converts "
    "dates and clocks to native types. It removed zero source rows in this release. [Cleaning notebook](../notebooks/04_Data_Cleaning.ipynb)\n"
    "- Validated many-to-one joins link transactions to batches and batches to medicine and manufacturer records. "
    "Only fixed catalogue storage information enters the features. The remaining lookup data support validation. "
    "[Integration report](data_integration.md)\n"
    "- The 16 audited inputs cover current observations, fixed storage information, and strictly earlier events of "
    "the same batch. The current row and same-timestamp rows contribute only to later history. "
    "Identifiers and timestamps remain metadata. [Leakage audit](leakage_audit.md)\n"
    "- Numerical values use training medians and standard scaling. Categorical values use training modes and "
    "one-hot encoding with unknowns ignored. The encoder is sparse internally. The combined release matrices use "
    "the smaller dense float32 representation. Training weights are saved separately. No split was resampled. "
    "[Fitting notebook](../notebooks/07_Final_Preprocessing.ipynb)",
    "## Saved split results",
]
rows_md = [
    "| Split | Rows | Engineered inputs | Transformed columns | Genuine | Fraud | Fraud share | Observed timestamp range |",
    "| --- | ---: | ---: | ---: | ---: | ---: | ---: | --- |",
]
for name, result in saved_splits.items():
    rows_md.append(
        f"| {name} | {result['rows']:,} | {len(features)} | {len(output_names)} | {result['genuine']:,} | "
        f"{result['fraud']:,} | {result['fraud']/result['rows']:.2%} | "
        f"`{result['first_timestamp']}` to `{result['last_timestamp']}` |"
    )
report.extend([
    "\n".join(rows_md),
    f"Raw engineered history is missing only for the {int(no_history.sum()):,} first-event rows. "
    "All transformed splits have zero missing and zero infinite values. All feature, label and timestamp rows "
    "round-tripped exactly. The model-ready files carry only encoded features. Keyed engineered and target "
    "files share their row order. See the output manifest for file hashes.",
    "## Verification and reproducibility",
    f"- Rebuilt the complete feature frame, independently recalculated all prior counts, and checked the detailed "
    f"history statistics against direct time filtering on {compared_history_rows} rows.\n"
    "- Found no duplicate composite keys or overlap between retained splits. Identifiers, outcomes, raw calendar "
    "values, and held-out simulator columns stayed outside the model matrix.\n"
    "- A fresh fit on training features reproduced the complete saved preprocessor state. Applying it to validation "
    "and test did not change that state. An unseen event type encoded to zeros for its field.\n"
    f"- Training class weights are {initial_weights['class_weights']['0']:.6f} for genuine and "
    f"{initial_weights['class_weights']['1']:.6f} for fraud.",
    "### Automated tests",
    "```text\n" + test_run.stdout.strip() + "\n```",
    "The release is suitable for retrospective preprocessing validation. Event arrival and fraud confirmation "
    "times are unavailable. Manufacturers and products recur across splits, so later performance claims about "
    "unseen groups require separate evaluation. [Leakage audit](leakage_audit.md)",
])
(reports / "preprocessing_report.md").write_text("\n\n".join(report) + "\n", encoding="utf-8")
print("Saved reports/preprocessing_report.md.")